# Huấn luyện chính thức CIFAR-10

## 1. Đọc nguồn cấu hình và chọn model
Vai trò: dùng duy nhất JSON đã chốt, đọc bằng chứng Phase 2, chọn một hoặc hai model. Mỗi lần chạy lại cell này tạo nhóm run mới; không sửa trực tiếp hyperparameter trong notebook.

In [5]:
from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone, timedelta
import gc
import hashlib
import json
import logging
import os
import shutil
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src/cifar_compare/train.py").is_file()), None
)
if PROJECT_ROOT is None:
    raise RuntimeError("Không tìm thấy project.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torchvision
from torchvision.datasets import CIFAR10
from torch.utils.data import RandomSampler, SequentialSampler
from src.cifar_compare.data import get_dataloaders, get_data_info
from src.cifar_compare.train import DEFAULTS, validate_config, train_model
from src.cifar_compare.utils import load_config, resolve_device, set_seed

CONFIG_PATH = PROJECT_ROOT / "configs/default.json"
REPORTS = PROJECT_ROOT / "reports"
base_config = load_config(CONFIG_PATH, DEFAULTS)
validate_config(base_config)
confirmations = json.loads((REPORTS / "protocol_confirmations.json").read_text(encoding="utf-8-sig"))
phase2 = json.loads((REPORTS / "phase2_readiness.json").read_text(encoding="utf-8-sig"))
environment = json.loads((REPORTS / "environment_phuc.json").read_text(encoding="utf-8-sig"))

def text_sha256(path):
    return hashlib.sha256(Path(path).read_text(encoding="utf-8-sig").encode("utf-8")).hexdigest()

MODELS_TO_TRAIN = ["resnet18"]
if not MODELS_TO_TRAIN or len(set(MODELS_TO_TRAIN)) != len(MODELS_TO_TRAIN):
    raise ValueError("Chọn ít nhất một model, không lặp tên.")
if not set(MODELS_TO_TRAIN).issubset({"resnet18", "convnext_tiny"}):
    raise ValueError("Chỉ hỗ trợ resnet18/convnext_tiny.")

SOURCE_FILES = [
    "src/cifar_compare/data.py", "src/cifar_compare/models.py",
    "src/cifar_compare/train.py", "src/cifar_compare/utils.py",
]
config_sha256 = text_sha256(CONFIG_PATH)
source_fingerprints = {name: text_sha256(PROJECT_ROOT / name) for name in SOURCE_FILES}
RUN_GROUP = datetime.now(timezone(timedelta(hours=8))).strftime("%Y%m%d_%H%M%S_%f")
planned_runs = {
    name: name + "_seed" + str(base_config["seed"]) + "_" + RUN_GROUP
    for name in MODELS_TO_TRAIN
}
ARTIFACT_ROOT = (PROJECT_ROOT / base_config["output_dir"]).resolve()
MIN_FREE_GIB = 5.0
preflight_passed = False

print("Project:", PROJECT_ROOT)
print("Kernel:", sys.executable)
print("Models:", MODELS_TO_TRAIN)
print("Config:", json.dumps(base_config, ensure_ascii=False, indent=2))
print("Planned run names:", json.dumps(planned_runs, indent=2))
print("Training chưa được bật.")

Project: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification
Kernel: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\.venv-1\Scripts\python.exe
Models: ['resnet18']
Config: {
  "seed": 42,
  "model": "convnext_tiny",
  "weights": "IMAGENET1K_V1",
  "pretrained": true,
  "freeze_backbone": false,
  "num_classes": 10,
  "data_root": "data/raw",
  "split_path": "data/splits/split_indices.json",
  "val_size": 5000,
  "input_size": 224,
  "batch_size": 32,
  "num_workers": 4,
  "epochs": 10,
  "learning_rate": 0.0001,
  "head_learning_rate": 0.001,
  "weight_decay": 0.05,
  "scheduler": "cosine",
  "amp": true,
  "early_stopping_patience": 5,
  "deterministic": false,
  "device": "cuda",
  "output_dir": "artifacts",
  "run_name": null,
  "eval_batch_size": 32,
  "warmup": 20,
  "repeats": 100
}
Planned run names: {
  "resnet18": "resnet18_seed42_20261117_175014_404116"
}
Tra

## 2. Đối chiếu protocol, môi trường và nơi lưu
Vai trò: phát hiện config/code/split đã đổi sau khi chốt; yêu cầu đúng CUDA và kernel đã kiểm chứng. Dung lượng tối thiểu 5 GiB là mức dự phòng cho cặp run hiện tại, không phải số dung lượng đã đo của checkpoint.

In [6]:
assert phase2["phase2_ready"], "Phase 2 chưa PASS."
assert config_sha256 == confirmations["config_sha256"] == phase2["config_sha256"], "Config đã đổi sau Phase 2."
assert source_fingerprints["src/cifar_compare/data.py"] == confirmations["data_py_sha256"], "data.py đã đổi."
assert confirmations["data_pipeline"]["status"] == "confirmed_via_user"
assert confirmations["evaluation_and_profiling"]["status"] == "confirmed_via_user"
assert torch.cuda.is_available(), "CUDA chưa khả dụng."
assert os.path.samefile(sys.executable, PROJECT_ROOT / ".venv-1/Scripts/python.exe"), "Chọn đúng kernel .venv-1."
assert str(torch.__version__) == environment["torch_version"], "Phiên bản torch đã đổi; ghi lại môi trường."
assert str(torchvision.__version__) == environment["torchvision_version"], "Phiên bản torchvision đã đổi."
DEVICE = resolve_device(base_config["device"])
assert DEVICE.type == "cuda"
gpu_name = torch.cuda.get_device_name(DEVICE)
assert "5070ti" in gpu_name.replace(" ", "").lower(), gpu_name

split_path = PROJECT_ROOT / base_config["split_path"]
assert text_sha256(split_path) == phase2["split_sha256"], "Split đã đổi sau Phase 2."
split = json.loads(split_path.read_text(encoding="utf-8-sig"))
assert split["seed"] == base_config["seed"] == 42
assert split["val_size"] == base_config["val_size"] == 5000
assert len(split["train_indices"]) == 45000 and len(split["val_indices"]) == 5000

if ARTIFACT_ROOT != (PROJECT_ROOT / "artifacts").resolve():
    raise RuntimeError("Protocol hiện tại ghi outputs tại artifacts; cập nhật protocol nếu đổi.")
for name, run_name in planned_runs.items():
    for subdir in ("checkpoints", "logs"):
        target = ARTIFACT_ROOT / subdir / run_name
        assert not target.exists(), "Run đã tồn tại, tạo run group mới: " + str(target)

free_gib = shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3)
assert free_gib >= MIN_FREE_GIB, "Cần thêm dung lượng dự phòng trước khi chạy."
print("PASS protocol/config/split/kernel/CUDA.")
print("GPU:", gpu_name)
print("Driver theo bản ghi T012:", environment["driver_query"]["output"])
print("Disk free GiB:", round(free_gib, 2), "| mức dự phòng:", MIN_FREE_GIB)
print("Output root:", ARTIFACT_ROOT)
print("Chưa tạo thư mục run/checkpoint và chưa tạo model.")

PASS protocol/config/split/kernel/CUDA.
GPU: NVIDIA GeForce RTX 5070 Ti
Driver theo bản ghi T012: 0, NVIDIA GeForce RTX 5070 Ti, 617.42, 16303 MiB
Disk free GiB: 94.47 | mức dự phòng: 5.0
Output root: C:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\artifacts
Chưa tạo thư mục run/checkpoint và chưa tạo model.


## 3. Kiểm tra DataLoader và lưu kết quả T015
Vai trò: dùng cấu hình vừa đọc để tạo loaders, in số mẫu/class order và kiểm tra một batch train, một batch validation. Test loader chỉ đọc metadata/số mẫu, **không duyệt test hoặc tính metric test**.

In [7]:
# Offline integrity check: không download hoặc lấy ảnh test để đánh giá.
offline_data = CIFAR10(base_config["data_root"], train=True, download=False)
del offline_data
set_seed(int(base_config["seed"]), bool(base_config["deterministic"]))
preflight_config = deepcopy(base_config)
check_train, check_val, check_test, class_names = get_dataloaders(preflight_config)
expected_classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
counts = {
    "train": len(check_train.dataset), "validation": len(check_val.dataset),
    "test": len(check_test.dataset),
}
assert counts == {"train": 45000, "validation": 5000, "test": 10000}, counts
assert class_names == expected_classes
assert isinstance(check_train.sampler, RandomSampler)
assert isinstance(check_val.sampler, SequentialSampler)
assert isinstance(check_test.sampler, SequentialSampler)
assert not check_val.drop_last and not check_test.drop_last

batch_results = {}
for split_name, loader in (("train", check_train), ("validation", check_val)):
    assert loader.batch_size == int(preflight_config["batch_size"])
    assert loader.num_workers == int(preflight_config["num_workers"])
    iterator = iter(loader)
    images, labels = next(iterator)
    del iterator
    expected_shape = (
        int(preflight_config["batch_size"]), 3,
        int(preflight_config["input_size"]), int(preflight_config["input_size"])
    )
    assert tuple(images.shape) == expected_shape
    assert labels.shape == (int(preflight_config["batch_size"]),)
    assert images.dtype == torch.float32 and labels.dtype == torch.int64
    assert torch.isfinite(images).all().item()
    assert 0 <= labels.min().item() <= labels.max().item() < int(preflight_config["num_classes"])
    batch_results[split_name] = {
        "image_shape": list(images.shape), "label_shape": list(labels.shape),
        "image_dtype": str(images.dtype), "label_dtype": str(labels.dtype),
        "finite": True, "num_workers": loader.num_workers,
    }
    print("PASS", split_name, batch_results[split_name])
    del images, labels

data_info = get_data_info(preflight_config)
preflight_report = {
    "captured_at": datetime.now(timezone(timedelta(hours=8))).isoformat(timespec="seconds"),
    "protocol_id": confirmations["protocol_id"], "run_group": RUN_GROUP,
    "config": deepcopy(base_config), "config_sha256": config_sha256,
    "source_fingerprints": source_fingerprints,
    "split_sha256": text_sha256(split_path),
    "samples": counts, "class_names": class_names, "data_info": data_info,
    "batches": batch_results, "device": str(DEVICE), "gpu_name": gpu_name,
    "disk_free_gib": free_gib, "minimum_free_gib": MIN_FREE_GIB,
    "planned_runs": planned_runs, "output_root": str(ARTIFACT_ROOT),
    "preflight_passed": True, "training_started_during_preflight": False,
    "test_evaluated": False,
}
REPORTS.mkdir(parents=True, exist_ok=True)
preflight_path = REPORTS / ("phase3_preflight_" + RUN_GROUP + ".json")
preflight_path.write_text(
    json.dumps(preflight_report, ensure_ascii=False, indent=2), encoding="utf-8"
)
del check_train, check_val, check_test
gc.collect()
preflight_passed = True

print("Samples:", counts)
print("Class order:", class_names)
print("Planned run names:", planned_runs)
print("T015 PREFLIGHT: PASS — CHƯA TRAINING")
print("Đã lưu:", preflight_path)
print("Hãy Ctrl+S rồi gửi output cell này để rà soát T015.")

PASS train {'image_shape': [32, 3, 224, 224], 'label_shape': [32], 'image_dtype': 'torch.float32', 'label_dtype': 'torch.int64', 'finite': True, 'num_workers': 4}
PASS validation {'image_shape': [32, 3, 224, 224], 'label_shape': [32], 'image_dtype': 'torch.float32', 'label_dtype': 'torch.int64', 'finite': True, 'num_workers': 4}
Samples: {'train': 45000, 'validation': 5000, 'test': 10000}
Class order: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Planned run names: {'resnet18': 'resnet18_seed42_20261117_175014_404116'}
T015 PREFLIGHT: PASS — CHƯA TRAINING
Đã lưu: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\reports\phase3_preflight_20261117_175014_404116.json
Hãy Ctrl+S rồi gửi output cell này để rà soát T015.


## 4. Training chính thức

In [8]:
RUN_FULL_TRAINING = True

if RUN_FULL_TRAINING:
    if not preflight_passed:
        raise RuntimeError("Hoàn tất cell preflight trước khi training.")
    assert text_sha256(CONFIG_PATH) == config_sha256, "Config đã đổi, làm lại preflight."
    assert text_sha256(split_path) == phase2["split_sha256"]
    for name, digest in source_fingerprints.items():
        assert text_sha256(PROJECT_ROOT / name) == digest, "Code đã đổi: " + name
    for run_name in planned_runs.values():
        for subdir in ("checkpoints", "logs"):
            assert not (ARTIFACT_ROOT / subdir / run_name).exists(), "Không ghi đè run cũ."
    assert shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3) >= MIN_FREE_GIB
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    training_results = {}
    result_path = REPORTS / ("training_runs_" + RUN_GROUP + ".json")
    for model_name in MODELS_TO_TRAIN:
        gc.collect()
        torch.cuda.empty_cache()
        run_config = deepcopy(base_config)
        run_config["model"] = model_name
        run_config["run_name"] = planned_runs[model_name]
        set_seed(int(run_config["seed"]), bool(run_config["deterministic"]))
        fresh_train, fresh_val, unused_test, fresh_classes = get_dataloaders(run_config)
        assert len(fresh_train.dataset) == 45000 and len(fresh_val.dataset) == 5000
        assert fresh_classes == expected_classes
        print("START:", model_name, run_config["run_name"])
        summary = train_model(
            model_name, run_config, fresh_train, fresh_val, fresh_classes, device=DEVICE
        )
        training_results[model_name] = summary
        result_path.write_text(
            json.dumps({
                "run_group": RUN_GROUP, "config_sha256": config_sha256,
                "results": training_results, "test_evaluated": False,
            }, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        del fresh_train, fresh_val, unused_test
        gc.collect()
        torch.cuda.empty_cache()
        print("DONE:", model_name, summary)
    print("Kết quả run:", result_path)
else:
    print("SKIP training: RUN_FULL_TRAINING=False. Chỉ chuẩn bị/T015, không chạy epoch.")

START: resnet18 resnet18_seed42_20261117_175014_404116


2026-11-17 17:50:52,013 | INFO | Model=resnet18 device=cuda AMP=True
2026-11-17 17:50:52,014 | INFO | Parameters total=11181642 trainable=11181642
2026-11-17 17:52:22,352 | INFO | Epoch 1/10 train loss=0.3832 acc=0.8686 val loss=0.2450 acc=0.9186 time=90.3s
2026-11-17 17:53:27,913 | INFO | Epoch 2/10 train loss=0.2103 acc=0.9282 val loss=0.1948 acc=0.9318 time=65.3s
2026-11-17 17:54:30,549 | INFO | Epoch 3/10 train loss=0.1563 acc=0.9469 val loss=0.1925 acc=0.9402 time=64.9s
2026-11-17 17:55:37,636 | INFO | Epoch 4/10 train loss=0.1189 acc=0.9585 val loss=0.1878 acc=0.9426 time=66.8s
2026-11-17 17:56:42,976 | INFO | Epoch 5/10 train loss=0.0873 acc=0.9704 val loss=0.1788 acc=0.9500 time=65.1s
2026-11-17 17:57:38,997 | INFO | Epoch 6/10 train loss=0.0585 acc=0.9802 val loss=0.1638 acc=0.9540 time=55.8s
2026-11-17 17:58:34,925 | INFO | Epoch 7/10 train loss=0.0385 acc=0.9872 val loss=0.1462 acc=0.9590 time=55.7s
2026-11-17 17:59:31,137 | INFO | Epoch 8/10 train loss=0.0231 acc=0.9927 val

DONE: resnet18 {'model': 'resnet18', 'best_epoch': 9, 'best_val_accuracy': 0.9646, 'best_val_loss': 0.13730251471996308, 'epochs_completed': 10, 'best_checkpoint': 'artifacts\\checkpoints\\resnet18_seed42_20261117_175014_404116\\best.pt', 'last_checkpoint': 'artifacts\\checkpoints\\resnet18_seed42_20261117_175014_404116\\last.pt', 'metrics_csv': 'artifacts\\logs\\resnet18_seed42_20261117_175014_404116\\metrics.csv', 'device': 'cuda'}
Kết quả run: c:\Users\havuh\OneDrive\Desktop\HocTap\ConvNeXtiny_vs_ResNet18_A_Comparative_Study_on_CIFAR10_Classification\reports\training_runs_20261117_175014_404116.json
